# BTC/USDT Chronos 시계열 예측 실험 (Colab GPU)
- Amazon Chronos-T5-Small (46M params)
- 롤링 윈도우로 6시간 후 가격 예측 → 매수/매도 시그널

**사용법**: 런타임 → 런타임 유형 변경 → GPU 선택 → 전체 실행

In [ ]:
# 1) 패키지 설치
!pip install -q ccxt chronos-forecasting stockstats seaborn

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from datetime import datetime
import ccxt
from stockstats import StockDataFrame as Sdf
from chronos import ChronosPipeline
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# 2) 설정
# ============================================================
PAIR = "BTC/USDT"
TIMEFRAME = "1h"
TRAIN_START = "2020-01-01"   # 컨텍스트용
TEST_START = "2024-06-01"
DATA_END = "2025-01-01"

# Chronos 설정
CHRONOS_MODEL = "amazon/chronos-t5-small"  # GPU니까 small 가능!
CONTEXT_LEN = 512      # 과거 512시간(~21일)
HORIZON = 6            # 6시간 후 예측
NUM_SAMPLES = 20       # 확률 샘플 수
PREDICT_STEP = 1       # 매 시간 예측

# 백테스트
INITIAL_BALANCE = 100_000
TRADING_FEE = 0.001

print("설정 완료")

In [ ]:
# ============================================================
# 3) 바이낸스 데이터 수집
# ============================================================
def fetch_btc_data(pair, timeframe, start_date, end_date):
    exchange = ccxt.binance({"enableRateLimit": True})
    since = exchange.parse8601(f"{start_date}T00:00:00Z")
    end_ts = exchange.parse8601(f"{end_date}T00:00:00Z")
    all_ohlcv = []
    print(f"[수집] {pair} {timeframe} | {start_date} ~ {end_date}")
    while since < end_ts:
        ohlcv = exchange.fetch_ohlcv(pair, timeframe=timeframe, since=since, limit=1000)
        if not ohlcv:
            break
        all_ohlcv.extend(ohlcv)
        since = ohlcv[-1][0] + 1
    df = pd.DataFrame(all_ohlcv, columns=["timestamp", "open", "high", "low", "close", "volume"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df[df["timestamp"] <= end_ts].drop_duplicates(subset=["timestamp"]).reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(np.float64)
    print(f"[완료] {len(df)}개 캔들")
    return df

raw = fetch_btc_data(PAIR, TIMEFRAME, TRAIN_START, DATA_END)
raw.head()

In [ ]:
# ============================================================
# 4) 테스트 구간 준비
# ============================================================
raw["date"] = pd.to_datetime(raw["date"])
test_df = raw[raw["date"] >= TEST_START].reset_index(drop=True)

test_start_idx = raw[raw["date"] >= TEST_START].index[0]
context_start = max(0, test_start_idx - CONTEXT_LEN)
full_prices = raw["close"].values[context_start:]
test_offset = test_start_idx - context_start

print(f"전체 가격 배열: {len(full_prices)}개")
print(f"테스트: {len(test_df)}개 ({TEST_START} ~ {DATA_END})")
print(f"테스트 시작 offset: {test_offset}")

In [ ]:
# ============================================================
# 5) Chronos 모델 로드 (GPU)
# ============================================================
print(f"모델 로딩: {CHRONOS_MODEL}")
pipeline = ChronosPipeline.from_pretrained(
    CHRONOS_MODEL,
    device_map=device,
    torch_dtype=torch.float32,
)
print("로딩 완료!")

In [ ]:
# ============================================================
# 6) 롤링 예측
# ============================================================
n = len(full_prices)
predictions = np.full(n, np.nan)
pred_upper = np.full(n, np.nan)
pred_lower = np.full(n, np.nan)

total_steps = (n - CONTEXT_LEN - HORIZON) // PREDICT_STEP
print(f"롤링 예측: {total_steps}회 (context={CONTEXT_LEN}, horizon={HORIZON})")

from tqdm import tqdm

for i in tqdm(range(CONTEXT_LEN, n - HORIZON, PREDICT_STEP)):
    context = torch.tensor(full_prices[i - CONTEXT_LEN:i], dtype=torch.float32)
    forecast = pipeline.predict(
        context.unsqueeze(0),
        HORIZON,
        num_samples=NUM_SAMPLES,
    )
    samples = forecast[0].numpy()
    final_step = samples[:, -1]
    predictions[i] = np.median(final_step)
    pred_upper[i] = np.percentile(final_step, 90)
    pred_lower[i] = np.percentile(final_step, 10)

# 테스트 구간만 추출
test_preds = predictions[test_offset:]
test_upper = pred_upper[test_offset:]
test_lower = pred_lower[test_offset:]
test_prices = full_prices[test_offset:]

print(f"유효 예측: {np.sum(~np.isnan(test_preds))}개")

In [ ]:
# ============================================================
# 7) 시그널 생성 + 백테스트
# ============================================================
valid = ~np.isnan(test_preds)
signals = np.zeros(len(test_prices), dtype=int)
signals[valid] = (test_preds[valid] > test_prices[valid]).astype(int)

# 백테스트
balance = float(INITIAL_BALANCE)
holdings = 0.0
total_trades = 0
portfolio = [INITIAL_BALANCE]

for i in range(len(test_prices) - 1):
    price = test_prices[i]
    sig = signals[i] if valid[i] else -1

    if sig == 1 and holdings == 0:
        holdings = balance / price * (1 - TRADING_FEE)
        balance = 0
        total_trades += 1
    elif sig == 0 and holdings > 0:
        balance = holdings * price * (1 - TRADING_FEE)
        holdings = 0
        total_trades += 1

    portfolio.append(balance + holdings * test_prices[i + 1])

if holdings > 0:
    balance = holdings * test_prices[-1] * (1 - TRADING_FEE)
    portfolio[-1] = balance

final = portfolio[-1]
model_ret = (final - INITIAL_BALANCE) / INITIAL_BALANCE * 100
bh_ret = (test_prices[-1] / test_prices[0] - 1) * 100

pv = np.array(portfolio)
peak = np.maximum.accumulate(pv)
mdd = ((pv - peak) / peak).min() * 100
returns = np.diff(pv) / pv[:-1]
sharpe = returns.mean() / (returns.std() + 1e-8) * np.sqrt(365 * 24)

print(f"\n{'='*60}")
print(f"[결과] {CHRONOS_MODEL}")
print(f"  초기 자산:      ${INITIAL_BALANCE:,.0f}")
print(f"  최종 자산:      ${final:,.0f}")
print(f"  Chronos 수익률: {model_ret:+.2f}%")
print(f"  Buy&Hold:       {bh_ret:+.2f}%")
print(f"  MDD:            {mdd:.2f}%")
print(f"  Sharpe:         {sharpe:.2f}")
print(f"  총 거래:        {total_trades}회")
print(f"{'='*60}")

In [ ]:
# ============================================================
# 8) 시각화
# ============================================================
fig, axes = plt.subplots(3, 1, figsize=(16, 12),
                         gridspec_kw={"height_ratios": [3, 2, 1]})

# 1) 포트폴리오
ax = axes[0]
pv_norm = pv / INITIAL_BALANCE
bh_norm = test_prices / test_prices[0]
ax.plot(pv_norm, label=f"Chronos ({model_ret:+.2f}%)", linewidth=2, color="#2196F3")
ax.plot(bh_norm, label=f"Buy&Hold ({bh_ret:+.2f}%)", linewidth=2, color="#FF9800", alpha=0.8)
ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5)
ax.set_title(f"Chronos: BTC/USDT Trading ({CHRONOS_MODEL})", fontsize=14, fontweight="bold")
ax.set_ylabel("Portfolio (normalized)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 2) 가격 + 예측
ax = axes[1]
ax.plot(test_prices, label="Actual", linewidth=1, color="#333", alpha=0.8)
idx = np.where(valid)[0]
ax.plot(idx, test_preds[valid], label="Predicted (median)", linewidth=0.8, color="#2196F3", alpha=0.6)
ax.fill_between(idx, test_lower[valid], test_upper[valid], alpha=0.15, color="#2196F3", label="80% CI")
ax.set_title("Price Prediction vs Actual", fontsize=12, fontweight="bold")
ax.set_ylabel("Price (USDT)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 3) Drawdown
ax = axes[2]
drawdown = (pv - peak) / peak * 100
ax.fill_between(range(len(drawdown)), drawdown, 0, alpha=0.4, color="#E53935")
ax.set_title(f"Drawdown (MDD: {mdd:.2f}%)", fontsize=12, fontweight="bold")
ax.set_xlabel("Step")
ax.set_ylabel("%")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("chronos_btc_result.png", dpi=150)
plt.show()
print("차트 저장: chronos_btc_result.png")

In [ ]:
# ============================================================
# 9) 결과 다운로드 (선택)
# ============================================================
# 예측 결과 CSV 저장
result_df = pd.DataFrame({
    "price": test_prices[:len(test_preds)],
    "prediction": test_preds,
    "pred_upper": test_upper,
    "pred_lower": test_lower,
    "signal": signals[:len(test_preds)],
})
result_df.to_csv("chronos_predictions.csv", index=False)

# Colab에서 다운로드
try:
    from google.colab import files
    files.download("chronos_btc_result.png")
    files.download("chronos_predictions.csv")
except:
    print("로컬 환경 - 파일이 현재 디렉토리에 저장됨")